xgboost, svc, knn, lda, qda, guassian nb, rf

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('/Users/tliu/Desktop/Erdos Project/3_Player_Data_Generation/match_data_50_tourns_modified.csv')
data.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p1_frames_played_3_years,p1_frames_won_3_years,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage
0,Cao Yupeng,Jiang Jun,9,1411,1044,0.918016,0.714532,341,179,2302,...,446,242,32,17,32,17,5,0,0.0,1.000000
1,Siripaporn Nuanthakhamjan,Zhou Yuelong,9,965,1384,0.056881,0.259705,3,0,23,...,23,4,245,118,808,430,2,5,1.0,0.285714
2,Wu Yize,Allan Taylor,9,1342,1237,0.656987,0.565251,78,37,552,...,404,215,101,49,336,147,5,3,0.0,0.625000
3,Ben Woollaston,Oliver Brown,9,1412,1146,0.845318,0.660383,803,454,5053,...,534,282,91,35,235,97,5,2,0.0,0.714286
4,Andres Petrov,Mark Williams,9,1045,1611,0.017765,0.195447,51,18,304,...,167,61,281,170,1112,649,2,5,1.0,0.285714


In [3]:
#Add more features
data['p1_frames_win_rate'] = data['p1_frames_won']/ data['p1_frames_played']
data['p2_frames_win_rate'] = data['p2_frames_won']/ data['p2_frames_played']

data['p1_matches_win_rate'] = data['p1_matches_won']/ data['p1_matches_played']
data['p2_matches_win_rate'] = data['p2_matches_won']/ data['p2_matches_played']

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5058 entries, 0 to 5057
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   player1                   5058 non-null   object 
 1   player2                   5058 non-null   object 
 2   best_of                   5058 non-null   int64  
 3   player1_elo               5058 non-null   int64  
 4   player2_elo               5058 non-null   int64  
 5   elo_match_win_rate        5058 non-null   float64
 6   elo_frame_win_rate        5058 non-null   float64
 7   p1_matches_played         5058 non-null   int64  
 8   p1_matches_won            5058 non-null   int64  
 9   p1_frames_played          5058 non-null   int64  
 10  p1_frames_won             5058 non-null   int64  
 11  p2_matches_played         5058 non-null   int64  
 12  p2_matches_won            5058 non-null   int64  
 13  p2_frames_played          5058 non-null   int64  
 14  p2_frame

In [5]:
#p1_matches_win_rate and p2_matches_win_rate both have missing values
#We will fill them with 0.5
data.fillna(0.5, inplace = True)

In [ ]:
#Train test split
from sklearn.model_selection import train_test_split
data_train, data_test = train_test_split(data, 
                                        test_size = 0.2,
                                        shuffle = False)

In [7]:
#Create two dictionaries to record the scores.
win_perc_pred_scores = {}

match_result_pred_scores = {}

In [8]:
#Import metrics from sklearn.metrics
from sklearn.metrics import accuracy_score, root_mean_squared_error

In [9]:
#Create predictors and targets for training and test set
result_train = data_train['match_result']
win_perc_train = data_train['win_percentage']
#exclude players' names and match results.
X_train = data_train.drop(['match_result', 'win_percentage', 'player1', 'player2', 'score1', 'score2'], axis = 1)


result_test = data_test['match_result']
win_perc_test = data_test['win_percentage']
X_test = data_test.drop(['match_result', 'win_percentage', 'player1', 'player2', 'score1', 'score2'], axis = 1)

## Model1: Random Forest

In [10]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

n_estinamtors = list(range(100, 800, 100))

m_depth = list(range(3, 9))

n_1, n_2, m_1, m_2 = 0,0,0,0

score1_best = 100
score2_best = 0

for n in n_estinamtors: 
    print(f'Random Forest: n_estimators = {n}')
    for m in m_depth:
        RFR = RandomForestRegressor(n_estimators = n, max_depth = m)

        RFR.fit(X_train, win_perc_train)
        pred = RFR.predict(X_test)
        score1 = root_mean_squared_error(pred, win_perc_test)
        if score1 < score1_best:
            score1_best = score1
            n_1 = n
            m_1 = m

        RFC = RandomForestClassifier(n_estimators = n, max_depth = m)
        RFC.fit(X_train, result_train)
        pred2 = RFC.predict(X_test)
        score2 = accuracy_score(pred2, result_test)

        if score2 > score2_best:
            score2_best = score2
            n_2 = n
            m_2 = m

#Record the best scores
win_perc_pred_scores[f'RFR(n_estimators = {n}, max_depth = {m})'] = score1_best
match_result_pred_scores[f'RFC(n_estimators = {n}, max_depth = {m})'] = score2_best

Random Forest: n_estimators = 100
Random Forest: n_estimators = 200
Random Forest: n_estimators = 300
Random Forest: n_estimators = 400
Random Forest: n_estimators = 500


KeyboardInterrupt: 

In [ ]:
print(win_perc_pred_scores)
print(match_result_pred_scores)

{'RFR(n_estimators = 700, max_depth = 8)': 0.2683459152530406}
{'RFC(n_estimators = 700, max_depth = 8)': 0.6610671936758893}
